In [67]:
import os

print("je m'execute depuis :", os.getcwd())
print("le dossier data existe :", os.path.isdir("../data"))
print("le fichier existe     :", os.path.isfile("../data/restaurants.json"))
print("contenu de ../data    :", os.listdir("../data"))

je m'execute depuis : c:\Users\Benjo\Documents\Formation DATA\json-api-geocodage\notebooks
le dossier data existe : True
le fichier existe     : True
contenu de ../data    : ['restaurants.json', 'restaurants_geocodes.csv', 'villes.json']


In [68]:
import json

with open("../data/restaurants.json", encoding="utf-8") as f:
    data = json.load(f)

print(type(data))
print(data["ville"], "-", data["pays"])
print("nombre de restaurants :", len(data["restaurants"]))

<class 'dict'>
Catane - Italie
nombre de restaurants : 4


In [69]:
print(data["restaurants"][0]["nom"])
print(data["restaurants"][0]["notes"][0])

Gelateria Zio Pietro dal 1964
4.6


In [70]:
for r in data["restaurants"]:
    moyenne = sum(r["notes"]) / len(r["notes"])
    print(f"{r["nom"]:32} {moyenne:.2f}")

Gelateria Zio Pietro dal 1964    4.70
A Casa d'Amici                   4.45
La Bitta                         4.15
Steak House                      4.00


In [71]:
print(json.dumps(data["restaurants"][1], ensure_ascii=False, indent=2))

{
  "nom": "A Casa d'Amici",
  "adresse": "Via Fischetti, 14, 95131 Catania CT",
  "type": "trattoria",
  "notes": [
    4.4,
    4.5
  ]
}


In [72]:
import pandas as pd

df = pd.json_normalize(data)
print(df.shape)
print(df.columns.tolist())

(1, 4)
['ville', 'pays', 'releve', 'restaurants']


In [73]:
df = pd.json_normalize(data, record_path="restaurants")
print(df)

                             nom                                   adresse  \
0  Gelateria Zio Pietro dal 1964  Via Porta di Ferro, 47, 95131 Catania CT   
1                 A Casa d'Amici       Via Fischetti, 14, 95131 Catania CT   
2                       La Bitta    Via Acquicella Porto, 95121 Catania CT   
3                    Steak House   Via Porta di Ferro, 8, 95100 Catania CT   

        type                 notes  
0    glacier       [4.6, 4.8, 4.7]  
1  trattoria            [4.4, 4.5]  
2    poisson  [4.2, 4.0, 4.3, 4.1]  
3      grill            [3.9, 4.1]  


In [74]:
df = pd.json_normalize(data, record_path="restaurants", meta=["ville", "pays"])
print(df)

                             nom                                   adresse  \
0  Gelateria Zio Pietro dal 1964  Via Porta di Ferro, 47, 95131 Catania CT   
1                 A Casa d'Amici       Via Fischetti, 14, 95131 Catania CT   
2                       La Bitta    Via Acquicella Porto, 95121 Catania CT   
3                    Steak House   Via Porta di Ferro, 8, 95100 Catania CT   

        type                 notes   ville    pays  
0    glacier       [4.6, 4.8, 4.7]  Catane  Italie  
1  trattoria            [4.4, 4.5]  Catane  Italie  
2    poisson  [4.2, 4.0, 4.3, 4.1]  Catane  Italie  
3      grill            [3.9, 4.1]  Catane  Italie  


In [75]:
#Challenge 1

#Etape 1

import json

with open("../data/villes.json", encoding="utf-8") as f:
    villes = json.load(f)

print(type(villes))
print("nombre de villes :", len(villes["villes"]))

print(villes["villes"][1]["nom"])
print(f"Nombre de restaurants à Syracuse : {len(villes["villes"][2]["restaurants"])}")
print(f"Nom du premier restaurant de Palerme : {villes["villes"][1]["restaurants"][0]["nom"]}")

<class 'dict'>
nombre de villes : 3
Palerme
Nombre de restaurants à Syracuse : 2
Nom du premier restaurant de Palerme : Antica Focacceria


In [76]:
# Etape 2

for r in villes["villes"]:
    print(f"Ville : {r["nom"]}, nombre de restaurants : {len(r["restaurants"])}")

Ville : Catane, nombre de restaurants : 3
Ville : Palerme, nombre de restaurants : 2
Ville : Syracuse, nombre de restaurants : 2


In [77]:
# Etape 3

df_villes = pd.json_normalize(villes, record_path=["villes", "restaurants"], meta=[["villes", "nom"], ["villes", "region"], ["villes","habitants"]])
print(df_villes)

                             nom       type                 notes villes.nom  \
0  Gelateria Zio Pietro dal 1964    glacier       [4.6, 4.8, 4.7]     Catane   
1                 A Casa d'Amici  trattoria            [4.4, 4.5]     Catane   
2                       La Bitta    poisson  [4.2, 4.0, 4.3, 4.1]     Catane   
3              Antica Focacceria  trattoria       [4.3, 4.1, 4.4]    Palerme   
4               Ferro di Cavallo  trattoria            [4.5, 4.6]    Palerme   
5                    Don Camillo    poisson  [4.7, 4.8, 4.6, 4.9]   Syracuse   
6               Gelateria Bianca    glacier                 [4.2]   Syracuse   

  villes.region villes.habitants  
0        Sicile           298762  
1        Sicile           298762  
2        Sicile           298762  
3        Sicile           630828  
4        Sicile           630828  
5        Sicile           121605  
6        Sicile           121605  


In [78]:
# Etape 4

df_villes["note_moyenne"] = [round(sum(r) / len(r), 2) for r in df_villes["notes"]]
df_villes

,nom,type,notes,villes.nom,villes.region,villes.habitants,note_moyenne
0,Gelateria Zio Pietro dal 1964,glacier,"[4.6, 4.8, 4.7]",Catane,Sicile,298762,4.70
1,A Casa d'Amici,trattoria,"[4.4, 4.5]",Catane,Sicile,298762,4.45
2,La Bitta,poisson,"[4.2, 4.0, 4.3, 4.1]",Catane,Sicile,298762,4.15
3,Antica Focacceria,trattoria,"[4.3, 4.1, 4.4]",Palerme,Sicile,630828,4.27
4,Ferro di Cavallo,trattoria,"[4.5, 4.6]",Palerme,Sicile,630828,4.55
5,Don Camillo,poisson,"[4.7, 4.8, 4.6, 4.9]",Syracuse,Sicile,121605,4.75
6,Gelateria Bianca,glacier,[4.2],Syracuse,Sicile,121605,4.20


In [79]:
# Quelle ville a la meilleure note moyenne ?

df_villes.groupby("villes.nom")["note_moyenne"].mean().sort_values(ascending=False).head(1)

villes.nom
Syracuse    4.475
Name: note_moyenne, dtype: float64

In [80]:
# Combien de restaurants de chaque type ?
df_villes["type"].value_counts()

type
trattoria    3
glacier      2
poisson      2
Name: count, dtype: int64

In [81]:
import requests

url = "https://api-adresse.data.gouv.fr/search/"
r = requests.get(url, params={"q": "20 avenue de Ségur 75007 Paris", "limit": 1})

print(r.status_code)

200


In [82]:
reponse = r.json()
print(type(reponse))
print(reponse.keys())

<class 'dict'>
dict_keys(['type', 'features', 'query'])


In [83]:
r = requests.get(url, params={"q": "20 avenue de Ségur 75007 Paris", "limit": 1})
print(r.url)

https://api-adresse.data.gouv.fr/search/?q=20+avenue+de+S%C3%A9gur+75007+Paris&limit=1


In [84]:
headers = {"User-Agent": "bootcamp-simplon-da/1.0 (prenom.nom@exemple.fr)"}
r = requests.get(url, params={"q": "Catane"}, headers=headers)

print(r.status_code)
print(r.json()["features"][0]["properties"]["label"])

200
Catane 31310 Montesquieu-Volvestre


In [85]:
import requests

def interroger(url, params, headers=None):
    try:
        r = requests.get(url, params=params, headers=headers, timeout=10)
        r.raise_for_status()
        return r.json()
    except requests.exceptions.Timeout:
        print("Trop lent, on abandonne cette requête")
    except requests.exceptions.HTTPError as e:
        print("Le serveur a répondu une erreur :", e)
    except requests.exceptions.RequestException as e:
        print("Problème réseau :", e)
    return None

In [86]:
print(interroger("https://api-adresse.data.gouv.fr/search/", {"q": "Lille", "limit": 1}) is not None)
print(interroger("https://api-adresse.data.gouv.fr/nimportequoi", {"q": "Lille"}))

True
Le serveur a répondu une erreur : 404 Client Error: Not Found for url: https://api-adresse.data.gouv.fr/nimportequoi?q=Lille
None


In [87]:
adresses = [
    "20 avenue de Ségur 75007 Paris",
    "Grand Place, Lille",
    "1 rue de la Paix, Paris",
    "place Bellecour, Lyon",
    "zzzzz qwerty 99999",
]

In [88]:
r = requests.get("https://api-adresse.data.gouv.fr/search/", params={"q": adresses[0], "limit":1}, headers=headers, timeout=10)
print(r.status_code)
print(r.json()["features"][0]["properties"]["label"])
print(r.json()["features"][0]["properties"]["score"])

200
20 Avenue de Ségur 75007 Paris
0.9716281818181818


In [89]:
resultat = []

for adresse in adresses:
    r = requests.get("https://api-adresse.data.gouv.fr/search/", params={"q": adresse, "limit":1}, headers=headers, timeout=10)
    if not r.json()["features"]:
        label = None
        score = None
        postcode = None
        city = None
        resultat.append({"adresse": adresse, "label": label, "score": score, "postcode": postcode, "city": city})
    else:
        label = r.json()["features"][0]["properties"]["label"]
        score = r.json()["features"][0]["properties"]["score"]
        postcode = r.json()["features"][0]["properties"]["postcode"]
        city = r.json()["features"][0]["properties"]["city"]
        resultat.append({"adresse": adresse, "label": label, "score": score, "postcode": postcode, "city": city})

resultat

[{'adresse': '20 avenue de Ségur 75007 Paris',
  'label': '20 Avenue de Ségur 75007 Paris',
  'score': 0.9716281818181818,
  'postcode': '75007',
  'city': 'Paris'},
 {'adresse': 'Grand Place, Lille',
  'label': 'Place de Gand 59800 Lille',
  'score': 0.5682435064935065,
  'postcode': '59800',
  'city': 'Lille'},
 {'adresse': '1 rue de la Paix, Paris',
  'label': '1 Rue de la Paix 75002 Paris',
  'score': 0.9640599999999998,
  'postcode': '75002',
  'city': 'Paris'},
 {'adresse': 'place Bellecour, Lyon',
  'label': 'Place Bellecour 69002 Lyon',
  'score': 0.9673672727272726,
  'postcode': '69002',
  'city': 'Lyon'},
 {'adresse': 'zzzzz qwerty 99999',
  'label': None,
  'score': None,
  'postcode': None,
  'city': None}]

In [90]:
# Etape 3
 
df_adresses = pd.DataFrame(resultat)
print(df_adresses.shape)
df_adresses

(5, 5)


,adresse,label,score,postcode,city
0,20 avenue de Ségur 75007 Paris,20 Avenue de Ségur 75007 Paris,0.971628,75007,Paris
1,"Grand Place, Lille",Place de Gand 59800 Lille,0.568244,59800,Lille
2,"1 rue de la Paix, Paris",1 Rue de la Paix 75002 Paris,0.964060,75002,Paris
3,"place Bellecour, Lyon",Place Bellecour 69002 Lyon,0.967367,69002,Lyon
4,zzzzz qwerty 99999,NaN,NaN,NaN,NaN


In [91]:
# Etape 4

print(f"{df_adresses["label"].count()} adresses trouvées.")

print(f"Le score le plus faible : '{round(df_adresses.sort_values('score').iloc[0]['score'], 2)}' pour l'adresse : '{df_adresses.sort_values('score').iloc[0]['adresse']}'")
print("Cela s'explique par le degré de précision du label, qui ne correspond pas du tout à l'adresse initiale.")
print("Le label ne correspond pas donc pas.")

4 adresses trouvées.
Le score le plus faible : '0.57' pour l'adresse : 'Grand Place, Lille'
Cela s'explique par le degré de précision du label, qui ne correspond pas du tout à l'adresse initiale.
Le label ne correspond pas donc pas.


In [92]:
import requests

url = "https://api-adresse.data.gouv.fr/search/"
r = requests.get(url, params={"q": "20 avenue de Ségur 75007 Paris", "limit": 1})
reponse = r.json()

import json
print(json.dumps(reponse, ensure_ascii=False, indent=2)[:600])

{
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "geometry": {
        "type": "Point",
        "coordinates": [
          2.308628,
          48.850699
        ]
      },
      "properties": {
        "label": "20 Avenue de Ségur 75007 Paris",
        "score": 0.9716281818181818,
        "housenumber": "20",
        "id": "75107_8909_00020",
        "banId": "c575af14-f697-464c-b70b-914cedb239a4",
        "name": "20 Avenue de Ségur",
        "postcode": "75007",
        "citycode": "75107",
        "x": 649266.35,
        "y": 6861406.23,
        "city": 


In [93]:
f = reponse["features"][0]
lon, lat = f["geometry"]["coordinates"]
print("longitude  :", lon, " latitude  :", lat)
print("label      :", f["properties"]["label"])

longitude  : 2.308628  latitude  : 48.850699
label      : 20 Avenue de Ségur 75007 Paris


In [94]:
import pandas as pd

df = pd.json_normalize(reponse, record_path="features")
print(df.columns.tolist())

['type', 'geometry.type', 'geometry.coordinates', 'properties.label', 'properties.score', 'properties.housenumber', 'properties.id', 'properties.banId', 'properties.name', 'properties.postcode', 'properties.citycode', 'properties.x', 'properties.y', 'properties.city', 'properties.district', 'properties.context', 'properties.type', 'properties.importance', 'properties.depcode', 'properties.street', 'properties._type']


In [95]:
df["lon"] = df["geometry.coordinates"].str[0]
df["lat"] = df["geometry.coordinates"].str[1]
print(df[["properties.label", "lat", "lon"]])

                 properties.label        lat       lon
0  20 Avenue de Ségur 75007 Paris  48.850699  2.308628


In [96]:
def geocoder_fr(adresse):
    """Renvoie (latitude, longitude) pour une adresse française, ou (None, None)."""
    url = "https://api-adresse.data.gouv.fr/search/"
    try:
        r = requests.get(url, params={"q": adresse, "limit": 1}, timeout=10)
        r.raise_for_status()
        features = r.json()["features"]
        if not features:
            return None, None
        lon, lat = features[0]["geometry"]["coordinates"]
        return lat, lon
    except requests.exceptions.RequestException:
        return None, None

In [97]:
print(geocoder_fr("20 avenue de Ségur 75007 Paris"))
print(geocoder_fr("zzzzz qwerty 99999"))

(48.850699, 2.308628)
(None, None)


In [98]:
url = "https://nominatim.openstreetmap.org/search"
headers = {"User-Agent": "bootcamp-simplon-da/1.0 (prenom.nom@exemple.fr)"}
params = {"q": "54 Via Pietro Mascagni, Catania, Italy", "format": "json", "limit": 1}

r = requests.get(url, params=params, headers=headers, timeout=10)
resultats = r.json()

print(resultats[0]["lat"], resultats[0]["lon"])
print(resultats[0]["display_name"])

37.5111425 15.0950365
Via Pietro Mascagni, Picanello-Ognina-Barriera-Canalicchio, Catania, Sicilia, 95131, Italia


In [99]:
import pandas as pd
df = pd.json_normalize(resultats)
print(df[["lat", "lon", "display_name"]])

          lat         lon                                       display_name
0  37.5111425  15.0950365  Via Pietro Mascagni, Picanello-Ognina-Barriera...


In [100]:
import time

def geocoder_monde(adresse):
    """Renvoie (latitude, longitude) n'importe où dans le monde."""
    url = "https://nominatim.openstreetmap.org/search"
    headers = {"User-Agent": "bootcamp-simplon-da/1.0 (fontes.benjamin@gmail.com)"}
    params = {"q": adresse, "format": "json", "limit": 1}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=10)
        r.raise_for_status()
        resultats = r.json()
        if not resultats:
            return None, None
        return float(resultats[0]["lat"]), float(resultats[0]["lon"])
    except requests.exceptions.RequestException:
        return None, None
    finally:
        time.sleep(1)

In [101]:
debut = time.time()
print(geocoder_monde("54 Via Pietro Mascagni, Catania, Italy"))
print(geocoder_monde("Piazza del Duomo, Catania, Italy"))
print(f"durée totale : {time.time() - debut:.1f} s")

(37.5111425, 15.0950365)
(37.5024076, 15.0871898)
durée totale : 2.7 s


In [102]:
import folium

point = [37.5112, 15.0949]          # latitude, longitude — Catane
m = folium.Map(location=point, zoom_start=14)
folium.Marker(location=point, popup="Un bon restaurant").add_to(m)
m

In [103]:
# Challenge 3

# Etape 1

import json

with open("../data/restaurants.json", encoding="utf-8") as f:
    resto_catane = json.load(f)

df_resto_catane = pd.json_normalize(resto_catane, record_path="restaurants", meta=["ville", "pays"])
print(df_resto_catane.shape)
df_resto_catane


(4, 6)


,nom,adresse,type,notes,ville,pays
0,Gelateria Zio Pietro dal 1964,"Via Porta di Ferro, 47, 95131 Catania CT",glacier,"[4.6, 4.8, 4.7]",Catane,Italie
1,A Casa d'Amici,"Via Fischetti, 14, 95131 Catania CT",trattoria,"[4.4, 4.5]",Catane,Italie
2,La Bitta,"Via Acquicella Porto, 95121 Catania CT",poisson,"[4.2, 4.0, 4.3, 4.1]",Catane,Italie
3,Steak House,"Via Porta di Ferro, 8, 95100 Catania CT",grill,"[3.9, 4.1]",Catane,Italie


In [104]:
# Etape 2

df_resto_catane["note_moyenne"] = [round(sum(r) / len(r), 2) for r in df_resto_catane["notes"]]
df_resto_catane["type"] = df_resto_catane["type"].str.title()
df_resto_catane



,nom,adresse,type,notes,ville,pays,note_moyenne
0,Gelateria Zio Pietro dal 1964,"Via Porta di Ferro, 47, 95131 Catania CT",Glacier,"[4.6, 4.8, 4.7]",Catane,Italie,4.70
1,A Casa d'Amici,"Via Fischetti, 14, 95131 Catania CT",Trattoria,"[4.4, 4.5]",Catane,Italie,4.45
2,La Bitta,"Via Acquicella Porto, 95121 Catania CT",Poisson,"[4.2, 4.0, 4.3, 4.1]",Catane,Italie,4.15
3,Steak House,"Via Porta di Ferro, 8, 95100 Catania CT",Grill,"[3.9, 4.1]",Catane,Italie,4.00


In [105]:
# Etape 3

import time

def geocoder_monde(adresse):
    """Renvoie (latitude, longitude) n'importe où dans le monde."""
    url = "https://nominatim.openstreetmap.org/search"
    headers = {"User-Agent": "json-api-geocodage/1.0 (benjamin.fontes@gmail.com)"}
    params = {"q": adresse, "format": "json", "limit": 1}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=10)
        r.raise_for_status()
        resultats = r.json()
        if not resultats:
            return None, None
        return float(resultats[0]["lat"]), float(resultats[0]["lon"])
    except requests.exceptions.RequestException:
        return None, None
    finally:
        time.sleep(1)

In [106]:
coordonnees = df_resto_catane["adresse"].apply(geocoder_monde)
df_resto_catane["lat"] = coordonnees.str[0]
df_resto_catane["lon"] = coordonnees.str[1]
df_resto_catane

,nom,adresse,type,notes,ville,pays,note_moyenne,lat,lon
0,Gelateria Zio Pietro dal 1964,"Via Porta di Ferro, 47, 95131 Catania CT",Glacier,"[4.6, 4.8, 4.7]",Catane,Italie,4.70,37.502093,15.093195
1,A Casa d'Amici,"Via Fischetti, 14, 95131 Catania CT",Trattoria,"[4.4, 4.5]",Catane,Italie,4.45,37.506171,15.094322
2,La Bitta,"Via Acquicella Porto, 95121 Catania CT",Poisson,"[4.2, 4.0, 4.3, 4.1]",Catane,Italie,4.15,37.489440,15.079431
3,Steak House,"Via Porta di Ferro, 8, 95100 Catania CT",Grill,"[3.9, 4.1]",Catane,Italie,4.00,37.502933,15.093041


In [107]:
# Etape 4

import folium

# 1. Le centre = coordonnées du 1er restaurant
centre = [df_resto_catane["lat"].iloc[0], df_resto_catane["lon"].iloc[0]]

# 2. La carte vide
m = folium.Map(location=centre, zoom_start=15, tiles="Esri.WorldStreetMap")

# 3. Un marqueur par restaurant
for r, ligne in df_resto_catane.iterrows():
    folium.Marker(
        location=[ligne["lat"], ligne["lon"]],
        popup=f"{ligne['nom']} - note : {ligne['note_moyenne']}"
    ).add_to(m)

m

In [108]:
# Etape 5

df_resto_catane.to_csv("../data/restaurants_geocodes.csv", index=False)